# Smoke test (synthetic data only)

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all** (10–20 minutes). Uses no benchmark data, so it is allowed before the protocol is accepted. When it finishes, copy the output of the **last cell** and send it back.

In [ ]:
!rm -rf fewshot_ad
!git clone REPLACE_WITH_YOUR_GITHUB_URL fewshot_ad
%cd fewshot_ad
!pip install -q -r requirements.txt 2>&1 | grep -i -E 'error|conflict' || echo 'Install OK'

In [ ]:
import sys, torch, torchvision
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| torchvision', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to GPU')
import anomalib, open_clip; print('anomalib', anomalib.__version__, '| open_clip', open_clip.__version__)

In [ ]:
!python run.py --smoke --out smoke_basic

In [ ]:
!rm -rf smoke_nn
!python run.py --smoke --out smoke_nn --methods patchcore,patchcore_dinov2,padim,simclr_scratch_standard,simclr_scratch_mild,simclr_finetune_standard,simclr_finetune_mild,winclip

In [ ]:
import os, pandas as pd
print('==== SUMMARY TO SEND BACK ====')
if os.path.exists('smoke_nn/results.csv'):
    df = pd.read_csv('smoke_nn/results.csv')
    print('Runs completed per method:'); print(df.groupby('method').size().to_string())
    !python analyze.py --out smoke_nn > /dev/null && echo 'analyze.py OK'
else:
    print('No method completed a run.')
if os.path.exists('smoke_nn/errors.csv'):
    e = pd.read_csv('smoke_nn/errors.csv').drop_duplicates('method')
    for _, r in e.iterrows():
        print(f"\n---- {r['method']} FAILED ----\n{r['traceback']}")
else:
    print('\nNo errors: all methods ran.')